# Imports 

In [0]:
#init
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col, upper, lower, when, endswith, startswith, trim
from pyspark.sql.types import StringType, IntegerType, DataType

#Loading from bronze_schema

In [0]:
df = spark.table("workspace.bronze_schema.crm_prd_info")
df.display()

# Explore the data

In [0]:
# check for nullable columns
df.printSchema()

In [0]:
df.describe().display()
"""
    Some columns have missing values
"""

## Check prd_id for duplicates and null values

In [0]:
(
    df.select("*")
    .groupBy(col("prd_id"))
    .count()
    .where((col("count") > 1) | (col("prd_id").isNull()) )

    .display()

)


"""
No duplicate or null values in the prd_id
"""

## Check for whitespaces in strings


In [0]:
# Check for whitespaces in strings
for field in df.schema.fields:

    if isinstance(field.dataType, StringType):
        white_space_df = (
            df.select("*")
            .where(col(field.name).startswith(" ") | col(field.name).endswith(" "))
        )
        white_space_count = white_space_df.count()
        print(f"Column name {field.name} has {white_space_count} white space rows")


"""
    White spaces found in prd_line column
"""

## Check prd_cost for negative or zero values 


In [0]:
# Check prd_cost for negative values 

df.select("*").where(col("prd_cost") <= 0).display()

## Checking for distinct values in prd_line


In [0]:
df.select(col("prd_line")).distinct().display()

"""
    Note to business user
    Please specify the full meaning of the abv RSMT
"""

In [0]:
# Check of all prd_start_date is before prd_end_date
df.select("*").filter(col("prd_start_dt") > col("prd_end_dt")).display()
"""
    Date is wrong
    End date cannot come before the start date
"""

## Checking for null values

In [0]:
# Checking for null values 
for column in df.columns:
    
    null_df = df.select("*").where(col(column).isNull())

    null_count = null_df.count()
    print(f"Column {column} has null count of {null_count}")
    if null_count:
        null_df.display()

"""
    We have null values in some of the columns
"""


# Data Notes
- Some columns have missing values
- The prd_line column has white spaces
- All prd_cost are postive values above 0 which is good.
- Prd_line needs to be normalized and mapped.
- End date cannot come before the start date

# Trasformation

## Trimming string columns

In [0]:
for field in df.schema.fields:
    column = field.name
    column_type = field.dataType

    if isinstance(column_type, StringType):
        print(f"Trimming {column_type} column {column} data")
        df = df.withColumn(column, trim(col(column)))

df.display()

## Splitting prd_key into cat_id and prd_key according to bsuiness decision

In [0]:
df =(
        df
        .withColumn(
            "prd_key",
            F.regexp_replace(col("prd_key"), "-", "_")
        )
        .withColumn(
            "cat_id",
            F.substring(col("prd_key"), 1, 5)
        )
        .withColumn(
            "prd_key",
            F.substring(col("prd_key"), 7, F.length(col("prd_key")))
        )

    )

df.display()

## Map prd_line codes to descriptive values
1. "M" - Mountain
2. "R" - Road
3. "S" - Other Sales
4. "T" - Touring

In [0]:
df = (
        df.withColumn(
            "prd_line",
            F.when(upper(col("prd_line")) == "M", "Mountain")
            .when(upper(col("prd_line")) == "R", "Road")
            .when(upper(col("prd_line")) == "S", "Other Sales")
            .when(upper(col("prd_line")) == "T", "Touring")
            .otherwise("n/a")
        )

    )
df.display()

## Fixing prd_start_dt and prd_end_dt
Business users says that all end date comes before the next start date

In [0]:
df=(
    df.withColumn(
        "prd_end_dt",
        (
            F.lead(col("prd_start_dt"))
            .over(Window.partitionBy("prd_key")
            .orderBy(col("prd_start_dt"))) 
            -1
         ).alias("new_prd_end_dt")
    ).orderBy(col("prd_id"))
)
df.display()

## Replace null values in prd_cost to 0


In [0]:
df = df.fillna(0, subset="prd_cost")
df.display()

In [0]:
# Quick check for null values
for column in df.columns:
    null_df = df.select("*").where(col(column).isNull())

    null_count = null_df.count()

    if null_count:
        print(column)
        null_df.display()

"""
    Only the prd_end_dt has null values and that is fine because some prd_cost dont have an end date.
    This is a pratical example of Type "2" Slowly Changing Descent 
"""

## Mapping the column names to more readable column names

In [0]:
COLUMNS_REMAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date",
    "cat_id": "category_id",
}

In [0]:
# Rename columns with columnRenamed

df = df.withColumnsRenamed(COLUMNS_REMAP)

df.display()

# Writing to the silver_schema

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver_schema.crm_prd_info")

# Read saved table from the silver schema

In [0]:
silver_crm_prd_info = spark.table("workspace.silver_schema.crm_prd_info")
silver_crm_prd_info.display()